# 🚌 Suroboyo Bus — P5 Data Engineering Pipeline
**Big Data Project — Institut Teknologi Sepuluh Nopember**

**Tugas P5:**
- Polling bus position dari Klacak API
- Klasifikasi bus: `bergerak` / `mangkal` / `berhenti_rute`
- Hitung headway real (exclude mangkal) vs SPM Dishub 15 menit
- Feature engineering untuk ML model
- Output CSV siap pakai untuk P3/P4

**Output files:**
| File | Isi |
|---|---|
| `bus_timeseries.csv` | Raw posisi bus tiap polling |
| `koridor_summary.csv` | Headway + armada per koridor per snapshot |
| `feature_engineered.csv` | Dataset final siap ML |
| `speed_heatmap.csv` | Kecepatan rata-rata per koridor per jam |
| `spatial_distribution.csv` | Distribusi spasial armada |

## ⚙️ 0. Config
> **Ganti `DURATION_SEC`** sebelum run: `120` untuk test, `3600` untuk 1 jam penuh.

In [105]:
import requests, math, time, os, csv
import pandas as pd
from datetime import datetime

# ── Ganti sesuai kebutuhan ──────────────────────────────
INTERVAL_SEC  = 30    # poll tiap N detik
DURATION_SEC  = 1800   # 120 = test (4 poll) | 3600 = 1 jam (120 poll)
# ────────────────────────────────────────────────────────

N_POLLS       = DURATION_SEC // INTERVAL_SEC

BUS_FILE      = "bus_timeseries.csv"
KORIDOR_FILE  = "koridor_summary.csv"
FEATURE_FILE  = "feature_engineered.csv"

POOL_LOCATIONS = [
    {"nama": "Terminal Purabaya",  "lat": -7.351352, "lng": 112.725213},
    {"nama": "Terminal Benowo",    "lat": -7.23589, "lng": 112.60778},
    {"nama": "Terminal Joyoboyo", "lat": -7.298961, "lng": 112.736182},
    {"nama": "TIJ Intermoda",     "lat": -7.298961, "lng": 112.736182},
    {"nama": "TOW Osowilangun",   "lat": -7.218318, "lng": 112.653038},
    {"nama": "Terminal Bratang",  "lat": -7.29876, "lng": 112.76159},
    {"nama": "Terminal Manukan",  "lat": -7.2389, "lng": 112.6814},
    {"nama": "Kejawan Putih Tambak", "lat": -7.276458, "lng": 112.802087},
    {"nama": "Terminal Menanggal", "lat": -7.34162, "lng": 112.72259},
    {"nama": "Park and Ride Mayjen Sungkono", "lat": -7.290839, "lng": 112.715194},
    {"nama": "Terminal Keputih", "lat": -7.29471, "lng": 112.80176},
]
POOL_RADIUS_M   = 200
SPEED_THRESHOLD = 3    # km/h
SPM_HEADWAY     = 15   # menit — standar pelayanan minimal Dishub Surabaya

print(f"Config loaded — {N_POLLS} polls x {INTERVAL_SEC}s = {DURATION_SEC}s total")

Config loaded — 60 polls x 30s = 1800s total


## 🔌 1. Bootstrap — Connect ke Klacak API

In [106]:
print("Connecting to Klacak API...")

bootstrap  = requests.get("https://busmapapi.fly.dev/all").json()
api_url    = bootstrap["apiUrl"]

route_data = requests.get(
    "https://raw.githubusercontent.com/DoubleA4/busmapsby/main/routedata.json"
).json()

halte_data = requests.get(
    "https://raw.githubusercontent.com/DoubleA4/busmapsby/main/halte.json"
).json()["halte"]

print(f"Connected!")
print(f"  {len(route_data)} rute tersedia")
print(f"  {len(halte_data)} halte tersedia")
print(f"  API URL: {api_url}")

Connecting to Klacak API...
Connected!
  16 rute tersedia
  927 halte tersedia
  API URL: https://busmapapi.fly.dev


## 🛠️ 2. Helper Functions

In [107]:
def haversine(lat1, lon1, lat2, lon2):
    """Hitung jarak dua koordinat GPS dalam meter."""
    R  = 6371000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a  = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))


def classify_bus(bus):
    """
    Klasifikasi status bus:
      bergerak      : speed >= 3 km/h
      mangkal       : diam DAN berada di dekat terminal/pool (radius 200m)
      berhenti_rute : diam tapi tidak di terminal (halte, lampu merah, macet)
    """
    speed = float(bus.get("speed", 0))
    if speed >= SPEED_THRESHOLD:
        return "bergerak"
    blat = float(bus.get("lat", 0))
    blng = float(bus.get("lng", 0))
    for pool in POOL_LOCATIONS:
        if haversine(blat, blng, pool["lat"], pool["lng"]) <= POOL_RADIUS_M:
            return "mangkal"
    return "berhenti_rute"


def get_req_addr(code):
    """Tentukan endpoint tracking berdasarkan kode koridor."""
    c = int(code)
    if c < 10 or c in (51, 12): return "sbybus"
    if c < 100:                  return "temanbus"
    return "feeder"


def save_csv(df, filepath, write_header):
    """Simpan CSV dengan QUOTE_ALL agar aman dari parsing error."""
    df.to_csv(
        filepath, mode="a",
        header=write_header,
        index=False,
        quoting=csv.QUOTE_ALL,
    )


def read_csv_safe(filepath):
    """Baca CSV dengan toleransi terhadap baris corrupt."""
    return pd.read_csv(
        filepath,
        quoting=csv.QUOTE_ALL,
        on_bad_lines="skip",
        engine="python",
    )

print("Helper functions loaded.")

Helper functions loaded.


## 🚌 3. Fetch Bus Positions

In [108]:
def fetch_all_buses():
    """
    Ambil posisi semua bus aktif dari semua koridor.
    Return: list of dict, satu entry per bus.
    """
    now  = datetime.now()
    ts   = now.strftime("%Y-%m-%d %H:%M:%S")
    hour = now.hour
    dow  = now.strftime("%A")
    rows = []

    for slug, route in route_data.items():
        code      = route["code"]
        token_str = bootstrap.get(code)
        if not token_str:
            continue
        token = token_str.split("/")[1] if "/" in token_str else token_str
        addr  = get_req_addr(code)

        try:
            resp = requests.get(
                f"{api_url}/track/{addr}/{code}",
                headers={
                    "Authorization": f"Bearer {token}",
                    "Cache-Control": "no-cache",
                },
                timeout=5,
            )
            if not resp.ok:
                continue

            for bus in resp.json():
                status = classify_bus(bus)
                rows.append({
                    # Waktu
                    "timestamp":   ts,
                    "hour":        hour,
                    "day_of_week": dow,
                    "is_weekend":  dow in ["Saturday", "Sunday"],
                    "is_peak":     hour in list(range(6, 10)) + list(range(16, 20)),
                    # Rute
                    "route":       slug,
                    "route_name":  route.get("title", slug),
                    "koridor":     code,
                    "feeder":      route.get("feeder", False),
                    # Bus
                    "bus_id":      str(bus.get("info", "unknown")),
                    "lat":         float(bus.get("lat", 0)),
                    "lng":         float(bus.get("lng", 0)),
                    "speed":       float(bus.get("speed", 0)),
                    "direction":   float(bus.get("direction", 0)),
                    "status":      status,
                })
        except Exception:
            pass

    return rows

print("fetch_all_buses() defined.")

fetch_all_buses() defined.


## ⏱️ 4. Headway Calculator

In [109]:
def compute_headway(buses_by_route):
    """
    Hitung headway estimasi per koridor.

    Formula  : headway = cycle_time / n_bus_efektif
    Cycle    : 120 menit (trunk) | 60 menit (feeder)
    Efektif  : total bus - bus mangkal
    SPM ref  : headway <= 15 menit (Dishub Surabaya)
    """
    rows = []

    for slug, buses in buses_by_route.items():
        route = route_data.get(slug, {})
        cycle = 60 if route.get("feeder") else 120

        n_total    = len(set(b["bus_id"] for b in buses))
        n_mangkal  = sum(1 for b in buses if b["status"] == "mangkal")
        n_berhenti = sum(1 for b in buses if b["status"] == "berhenti_rute")
        n_bergerak = sum(1 for b in buses if b["status"] == "bergerak")
        n_efektif  = n_total - n_mangkal

        hw_raw  = round(cycle / n_total,   1) if n_total   > 0 else None
        hw_real = round(cycle / n_efektif, 1) if n_efektif > 0 else None
        avg_spd = round(sum(b["speed"] for b in buses) / len(buses), 1) if buses else 0
        pct_eff = round(n_efektif / n_total * 100, 1) if n_total > 0 else 0
        hw_gap  = round(hw_real - SPM_HEADWAY, 1) if hw_real else None

        if hw_real is None:      svc = "tidak_beroperasi"
        elif hw_real <= 15:      svc = "baik"
        elif hw_real <= 25:      svc = "sedang"
        else:                    svc = "buruk"

        rows.append({
            "timestamp":          buses[0]["timestamp"],
            "hour":               buses[0]["hour"],
            "day_of_week":        buses[0]["day_of_week"],
            "is_weekend":         buses[0]["is_weekend"],
            "is_peak":            buses[0]["is_peak"],
            "route":              slug,
            "route_name":         route.get("title", slug),
            "feeder":             route.get("feeder", False),
            "cycle_time_min":     cycle,
            "n_total":            n_total,
            "n_mangkal":          n_mangkal,
            "n_berhenti_rute":    n_berhenti,
            "n_bergerak":         n_bergerak,
            "n_efektif":          n_efektif,
            "pct_efektif":        pct_eff,
            "headway_raw_min":    hw_raw,
            "headway_real_min":   hw_real,
            "headway_gap_vs_spm": hw_gap,
            "avg_speed_kmh":      avg_spd,
            "service_level":      svc,
        })

    return pd.DataFrame(rows).sort_values("headway_real_min")

print("compute_headway() defined.")

compute_headway() defined.


## 🔧 5. Feature Engineering

In [110]:
def feature_engineering(df_kor):
    """
    Tambah fitur turunan untuk ML model.
    Input : koridor_summary DataFrame
    Output: DataFrame dengan fitur tambahan + label demand_level
    """
    df = df_kor.copy()

    # Konversi tipe
    for col in ["hour", "n_total", "n_mangkal", "n_efektif",
                "headway_real_min", "avg_speed_kmh", "pct_efektif"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Kategori waktu
    df["time_category"] = pd.cut(
        df["hour"],
        bins   = [0, 6, 9, 12, 16, 20, 24],
        labels = ["dini_hari", "pagi_peak", "siang",
                  "sore_peak", "malam", "tengah_malam"],
        right  = False,
    ).astype(str)

    # Flag inefisiensi armada
    df["ada_mangkal"]   = df["n_mangkal"] > 0
    df["pct_mangkal"]   = (df["n_mangkal"] / df["n_total"].replace(0, 1) * 100).round(1)

    # Flag headway
    df["headway_ok"]     = df["headway_real_min"] <= SPM_HEADWAY
    df["headway_kritis"] = df["headway_real_min"] > 25

    # Label demand — TARGET VARIABLE untuk ML
    # Proxy: lebih banyak bus efektif = demand lebih tinggi
    df["demand_level"] = df["n_efektif"].apply(
        lambda x: "tinggi" if x >= 12
        else "sedang"      if x >= 7
        else "rendah"
    )

    # Encode kategorik untuk ML
    df["feeder_enc"]     = df["feeder"].astype(int)
    df["is_peak_enc"]    = df["is_peak"].astype(int)
    df["is_weekend_enc"] = df["is_weekend"].astype(int)

    return df

print("feature_engineering() defined.")

feature_engineering() defined.


## 👀 6. Snapshot Preview
Cek dulu satu snapshot sebelum mulai polling loop.

In [111]:
print("Fetching snapshot pertama...")
buses = fetch_all_buses()
print(f"{len(buses)} bus terdeteksi\n")

by_route = {}
for b in buses:
    by_route.setdefault(b["route"], []).append(b)

df_hw = compute_headway(by_route)

n_mangkal_total = sum(1 for b in buses if b["status"] == "mangkal")
n_efektif_total = len(buses) - n_mangkal_total

print("=" * 80)
print("SNAPSHOT — ARMADA & HEADWAY PER KORIDOR")
print("=" * 80)
print(df_hw[[
    "route_name", "n_total", "n_mangkal", "n_efektif",
    "headway_raw_min", "headway_real_min",
    "avg_speed_kmh", "service_level",
]].to_string(index=False))

print(f"\nTotal bus  : {len(buses)}")
print(f"Mangkal    : {n_mangkal_total}")
print(f"Efektif    : {n_efektif_total}")

Fetching snapshot pertama...
89 bus terdeteksi

SNAPSHOT — ARMADA & HEADWAY PER KORIDOR
                 route_name  n_total  n_mangkal  n_efektif  headway_raw_min  headway_real_min  avg_speed_kmh service_level
          TIJ - Lakarsantri       19          1         18              3.2               3.3           24.5          baik
         TIJ - Gunung Anyar       13          1         12              4.6               5.0           21.0          baik
     Term. Benowo-Tunjungan       16          5         11              7.5              10.9           12.9          baik
              Kejawan-UNESA       15          5         10              8.0              12.0           11.4          baik
Mayjend Sungkono-Puspa Raya        7          2          5              8.6              12.0           13.6          baik
    Purabaya-UNAIR Kampus C       14          4         10              8.6              12.0           16.1          baik
Mayjend Sungkono-Balai Kota        5          2    

## 📡 7. Polling Loop
> **Pastikan DURATION_SEC sudah benar di cell Config sebelum run cell ini.**

In [112]:
# Hapus file lama — fresh start
for f in [BUS_FILE, KORIDOR_FILE]:
    if os.path.exists(f):
        os.remove(f)
        print(f"File lama dihapus: {f}")

print(f"\nMulai polling {N_POLLS}x | interval {INTERVAL_SEC}s | total {DURATION_SEC}s\n")

for i in range(N_POLLS):

    # Refresh token tiap 30 menit
    if i > 0 and i % 60 == 0:
        bootstrap = requests.get("https://busmapapi.fly.dev/all").json()
        print(f"   Token refreshed at poll #{i+1}")

    buses = fetch_all_buses()
    if not buses:
        print(f"[{i+1:03d}] No data returned, skipping...")
        time.sleep(INTERVAL_SEC)
        continue

    by_route = {}
    for b in buses:
        by_route.setdefault(b["route"], []).append(b)

    df_buses = pd.DataFrame(buses)
    df_kor   = compute_headway(by_route)

    write_header = (i == 0)
    save_csv(df_buses, BUS_FILE,     write_header)
    save_csv(df_kor,   KORIDOR_FILE, write_header)

    n_mang = sum(1 for b in buses if b["status"] == "mangkal")
    n_eff  = len(buses) - n_mang
    print(
        f"[{i+1:03d}] {buses[0]['timestamp']} | "
        f"{len(buses)} bus | {n_mang} mangkal | "
        f"{n_eff} efektif | {len(by_route)} koridor aktif"
    )

    time.sleep(INTERVAL_SEC)

print("\nPolling selesai!")

File lama dihapus: bus_timeseries.csv
File lama dihapus: koridor_summary.csv

Mulai polling 60x | interval 30s | total 1800s

[001] 2026-06-21 06:59:32 | 89 bus | 20 mangkal | 69 efektif | 7 koridor aktif
[002] 2026-06-21 07:00:09 | 151 bus | 39 mangkal | 112 efektif | 14 koridor aktif
[003] 2026-06-21 07:00:46 | 93 bus | 25 mangkal | 68 efektif | 10 koridor aktif
[004] 2026-06-21 07:01:25 | 155 bus | 43 mangkal | 112 efektif | 15 koridor aktif
[005] 2026-06-21 07:02:01 | 155 bus | 43 mangkal | 112 efektif | 15 koridor aktif
[006] 2026-06-21 07:02:38 | 155 bus | 44 mangkal | 111 efektif | 15 koridor aktif
[007] 2026-06-21 07:03:15 | 155 bus | 45 mangkal | 110 efektif | 15 koridor aktif
[008] 2026-06-21 07:03:52 | 157 bus | 46 mangkal | 111 efektif | 15 koridor aktif
[009] 2026-06-21 07:04:28 | 157 bus | 48 mangkal | 109 efektif | 15 koridor aktif
[010] 2026-06-21 07:05:05 | 155 bus | 48 mangkal | 107 efektif | 15 koridor aktif
[011] 2026-06-21 07:05:44 | 156 bus | 48 mangkal | 108 efek

## 📊 8. Analisis Dataset

In [113]:
print("ANALISIS DATASET")
print("=" * 80)

df_ts  = read_csv_safe(BUS_FILE)
df_kor = read_csv_safe(KORIDOR_FILE)

# Konversi tipe
for col in ["speed", "lat", "lng", "hour"]:
    df_ts[col] = pd.to_numeric(df_ts[col], errors="coerce")
for col in ["headway_real_min", "n_total", "n_mangkal", "n_efektif", "avg_speed_kmh"]:
    df_kor[col] = pd.to_numeric(df_kor[col], errors="coerce")

print(f"Records bus      : {len(df_ts):,}")
print(f"Records koridor  : {len(df_kor):,}")
print(f"Timespan         : {df_ts['timestamp'].min()} -> {df_ts['timestamp'].max()}")
print(f"Bus unik         : {df_ts['bus_id'].nunique()}")
print(f"Rute terekam     : {df_ts['route'].nunique()}")

ANALISIS DATASET
Records bus      : 9,058
Records koridor  : 873
Timespan         : 2026-06-21 06:59:32 -> 2026-06-21 07:36:03
Bus unik         : 158
Rute terekam     : 15


In [114]:
# Komposisi status bus per rute
print("KOMPOSISI STATUS BUS PER RUTE:")
status_tbl = (
    df_ts.groupby(["route_name", "status"])
    .size().unstack(fill_value=0)
)
print(status_tbl.to_string())

KOMPOSISI STATUS BUS PER RUTE:
status                             bergerak  berhenti_rute  mangkal
route_name                                                         
Kejawan-UNESA                           505             43      307
Mayjend Sungkono-Balai Kota             141              8      131
Mayjend Sungkono-Puspa Raya             283             19      161
Purabaya-ITS-Kenjeran Park              292             58      237
Purabaya-Perak                          330            118      281
Purabaya-UNAIR Kampus C                 468             63      253
SIER-Kota Lama                          267             73       66
TIJ - Gunung Anyar                      536            111      130
TIJ - Lakarsantri                       960             92       50
TOW - UNESA                             121             12      103
Term. Benowo-Tunjungan                  524             64      308
Term. Bratang - Stasiun Psr. Turi       228            115      184
Term. Keputih-Bun

In [115]:
# Speed heatmap (bus bergerak saja)
print("KECEPATAN RATA-RATA BUS BERGERAK PER KORIDOR PER JAM (km/h):")
df_moving   = df_ts[df_ts["status"] == "bergerak"]
speed_pivot = (
    df_moving.groupby(["route_name", "hour"])["speed"]
    .mean().round(1).unstack()
)
print(speed_pivot.to_string())
speed_pivot.reset_index().to_csv("speed_heatmap.csv", index=False, quoting=csv.QUOTE_ALL)
print("\nDisimpan: speed_heatmap.csv")

KECEPATAN RATA-RATA BUS BERGERAK PER KORIDOR PER JAM (km/h):
hour                                  6     7
route_name                                   
Kejawan-UNESA                      19.0  23.0
Mayjend Sungkono-Balai Kota        18.7  23.0
Mayjend Sungkono-Puspa Raya        19.0  24.1
Purabaya-ITS-Kenjeran Park          NaN  39.0
Purabaya-Perak                      NaN  27.3
Purabaya-UNAIR Kampus C            25.1  24.9
SIER-Kota Lama                      NaN  27.9
TIJ - Gunung Anyar                 24.8  22.0
TIJ - Lakarsantri                  25.9  24.4
TOW - UNESA                         NaN  31.4
Term. Benowo-Tunjungan             25.8  28.1
Term. Bratang - Stasiun Psr. Turi   NaN  26.4
Term. Keputih-Bunguran              NaN  25.5
Term. Menanggal-Term. Manukan       NaN  25.8
Terminal Bratang - Shelter Bulak    NaN  28.6

Disimpan: speed_heatmap.csv


In [116]:
# Headway stats per koridor
print("HEADWAY REAL PER KORIDOR (menit):")
hw_stats = (
    df_kor.groupby("route_name")["headway_real_min"]
    .agg(["mean", "min", "max"]).round(1)
)
hw_stats.columns = ["avg_headway", "best_headway", "worst_headway"]
hw_stats["vs_spm"] = hw_stats["avg_headway"].apply(
    lambda x: "OK" if x <= 15 else f"+{round(x-15,1)} mnt"
)
print(hw_stats.sort_values("avg_headway").to_string())

HEADWAY REAL PER KORIDOR (menit):
                                   avg_headway  best_headway  worst_headway     vs_spm
route_name                                                                            
TIJ - Lakarsantri                          3.3           3.2            3.5         OK
TIJ - Gunung Anyar                         5.4           5.0            6.0         OK
Term. Menanggal-Term. Manukan              7.0           6.7            8.6         OK
Purabaya-ITS-Kenjeran Park                10.2           8.6           12.0         OK
Term. Bratang - Stasiun Psr. Turi         10.4           8.6           12.0         OK
SIER-Kota Lama                            10.6           8.6           15.0         OK
Term. Benowo-Tunjungan                    11.5          10.0           13.3         OK
Mayjend Sungkono-Puspa Raya               12.1          10.0           15.0         OK
Kejawan-UNESA                             12.6          10.9           15.0         OK
Purabaya-

In [117]:
# Distribusi spasial (bus efektif saja)
df_eff  = df_ts[df_ts["status"] != "mangkal"]
spatial = df_eff.groupby("route_name").agg(
    lat_min  =("lat",   "min"),
    lat_max  =("lat",   "max"),
    lng_min  =("lng",   "min"),
    lng_max  =("lng",   "max"),
    avg_speed=("speed", "mean"),
).round(4)
spatial.to_csv("spatial_distribution.csv", quoting=csv.QUOTE_ALL)
print("Disimpan: spatial_distribution.csv")
print(spatial.to_string())

Disimpan: spatial_distribution.csv
                                   lat_min  lat_max   lng_min   lng_max  avg_speed
route_name                                                                        
Kejawan-UNESA                      -7.2996  -7.2558  112.6754  112.8086    21.1496
Mayjend Sungkono-Balai Kota        -7.2913  -7.2607  112.7141  112.7512    21.6711
Mayjend Sungkono-Puspa Raya        -7.2953  -7.2552  112.6305  112.7190    22.5132
Purabaya-ITS-Kenjeran Park         -7.3555  -7.2498  112.7118  112.7958    32.5657
Purabaya-Perak                     -7.3555   7.2984  112.7119  112.7456    20.1674
Purabaya-UNAIR Kampus C            -7.3556  -7.2634  112.7122  112.7978    21.9906
SIER-Kota Lama                     -7.3296  -7.2351  112.7340  112.7544    21.9441
TIJ - Gunung Anyar                 -7.3439  -7.2890  112.7334  112.8103    18.2380
TIJ - Lakarsantri                  -7.3141  -7.2873  112.6332  112.7395    22.2842
TOW - UNESA                        -7.2988  -7.1990 

## 🔧 9. Feature Engineering & Export

In [118]:
df_feat = feature_engineering(df_kor)
save_csv(df_feat, FEATURE_FILE, write_header=True)

print("DISTRIBUSI DEMAND LEVEL (target variable ML):")
print(df_feat["demand_level"].value_counts().to_string())

print("\nDISTRIBUSI SERVICE LEVEL:")
print(df_feat["service_level"].value_counts().to_string())

print("\nDISTRIBUSI TIME CATEGORY:")
print(df_feat["time_category"].value_counts().to_string())

DISTRIBUSI DEMAND LEVEL (target variable ML):
demand_level
rendah    459
sedang    332
tinggi     82

DISTRIBUSI SERVICE LEVEL:
service_level
baik      678
sedang    132
buruk      63

DISTRIBUSI TIME CATEGORY:
time_category
pagi_peak    873


In [119]:
print("OUTPUT FILES:")
print("-" * 60)
for f in [BUS_FILE, KORIDOR_FILE, FEATURE_FILE,
          "speed_heatmap.csv", "spatial_distribution.csv"]:
    if os.path.exists(f):
        n_rows = len(read_csv_safe(f))
        size   = os.path.getsize(f) // 1024
        print(f"  {f:<40} {n_rows:>6} rows | {size} KB")

print("\nKOLOM FEATURE_ENGINEERED.CSV (input ML):")
ml_cols = [
    "hour", "is_peak_enc", "is_weekend_enc", "feeder_enc",
    "time_category", "n_total", "n_efektif", "pct_efektif",
    "headway_real_min", "headway_gap_vs_spm", "headway_ok",
    "avg_speed_kmh", "ada_mangkal", "pct_mangkal",
    "service_level", "demand_level",
]
for c in ml_cols:
    dtype = str(df_feat[c].dtype) if c in df_feat.columns else "MISSING"
    print(f"  {c:<30} {dtype}")

print("\nPipeline P5 selesai!")

OUTPUT FILES:
------------------------------------------------------------
  bus_timeseries.csv                         9058 rows | 1399 KB
  koridor_summary.csv                         873 rows | 132 KB
  feature_engineered.csv                     1111 rows | 237 KB
  speed_heatmap.csv                            15 rows | 0 KB
  spatial_distribution.csv                     15 rows | 1 KB

KOLOM FEATURE_ENGINEERED.CSV (input ML):
  hour                           int64
  is_peak_enc                    int64
  is_weekend_enc                 int64
  feeder_enc                     int64
  time_category                  object
  n_total                        int64
  n_efektif                      int64
  pct_efektif                    float64
  headway_real_min               float64
  headway_gap_vs_spm             float64
  headway_ok                     bool
  avg_speed_kmh                  float64
  ada_mangkal                    bool
  pct_mangkal                    float64
  service_l